# Q2: Unsupervised Learning — Customer Segmentation

This notebook performs customer segmentation using K-Means clustering and visualises the results with PCA dimensionality reduction.

## 1. Data Preparation

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/q2_customers.csv')
print('Shape:', df.shape)
print('\nFeatures:', list(df.columns))
print('\nFirst 5 rows:')
print(df.head())
print('\nSummary statistics:')
print(df.describe())

Shape: (500, 6)

Features: ['age', 'annual_spend', 'visits_per_month', 'basket_size', 'days_since_last_visit', 'num_categories_purchased']

First 5 rows:
   age  annual_spend  visits_per_month  basket_size  days_since_last_visit  num_categories_purchased
0   30         43075                 5         3214                     45                         6
1   19         14496                 2         1832                      8                         3
2   43         57632                 6         4120                     16                         4
3   56         89210                 9         5890                      7                         8
4   24         22340                 3         2100                     32                         2

Summary statistics:
              age  annual_spend  visits_per_month  basket_size  days_since_last_visit  num_categories_purchased
count  500.00000    500.000000        500.000000   500.000000             500.000000                500.000

In [2]:
# Scale all features with StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)
X_scaled_df = pd.DataFrame(X_scaled, columns=df.columns)
print('Scaling complete. Sample of scaled data:')
print(X_scaled_df.head(3))

Scaling complete. Sample of scaled data:
        age  annual_spend  visits_per_month  basket_size  days_since_last_visit  num_categories_purchased
0 -0.726117     -0.236091         -0.107141    -0.208081               0.708285                  0.551858
1 -1.489268     -1.235248         -1.068591    -0.868988              -0.822150                 -0.680162
2  0.177308      0.272056          0.214225     0.225165              -0.490780                 -0.270576


**Why scaling is essential before K-Means:**

K-Means computes Euclidean distances between data points and centroids. Features with large numeric ranges (e.g., `annual_spend` up to ₹124,800) would dominate the distance calculations over features with smaller ranges (e.g., `visits_per_month` max = 12). StandardScaler transforms all features to have mean=0 and std=1, ensuring every feature contributes equally to cluster assignments.

## 2. Choosing K — Elbow Method

In [3]:
# Compute WCSS for K = 1 to 10
wcss = []
for k in range(1, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    wcss.append(km.inertia_)

print('WCSS values:')
for k, w in enumerate(wcss, 1):
    print(f'  K={k}: {w:.2f}')

plt.figure(figsize=(8, 5))
plt.plot(range(1, 11), wcss, marker='o', color='steelblue', linewidth=2, markersize=8)
plt.axvline(x=4, color='tomato', linestyle='--', label='Optimal K=4')
plt.title('Elbow Method — Within-Cluster Sum of Squares', fontsize=13, fontweight='bold')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('WCSS')
plt.legend()
plt.xticks(range(1, 11))
plt.tight_layout()
plt.savefig('elbow_plot.png', dpi=100, bbox_inches='tight')
plt.show()
print('Elbow plot saved.')

WCSS values:
  K=1: 3000.00
  K=2: 968.99
  K=3: 561.25
  K=4: 444.93
  K=5: 402.37
  K=6: 370.39
  K=7: 346.95
  K=8: 319.90
  K=9: 303.28
  K=10: 289.11

Elbow plot saved.


**Optimal K = 4**

The WCSS drops sharply from K=1 to K=3, and there is a noticeable change in the rate of decrease at K=4 — the point where adding another cluster yields diminishing returns. Beyond K=4, the improvement per additional cluster is relatively small and does not justify the added model complexity. Therefore, **K=4** is selected as the optimal number of clusters.

## 3. K-Means Clustering

In [4]:
# Fit K-Means with K=4
km_final = KMeans(n_clusters=4, random_state=42, n_init=10)
km_final.fit(X_scaled)
df['cluster'] = km_final.labels_

print('Cluster sizes:')
print(df['cluster'].value_counts().sort_index())

# Cluster centroids in scaled space
centroids_df = pd.DataFrame(
    km_final.cluster_centers_,
    columns=df.columns[:-1]
)
centroids_df.index.name = 'Cluster'
print('\nCluster Centroids (scaled):')
print(centroids_df.round(3))

print('\nCluster Means (original scale):')
print(df.groupby('cluster').mean().round(2))

Cluster sizes:
cluster
0    170
1     80
2    165
3     85
Name: count, dtype: int64

Cluster Centroids (scaled):
              age  annual_spend  visits_per_month  basket_size  days_since_last_visit  num_categories_purchased
Cluster                                                                                                         
0          -1.095        -1.036            -1.012       -1.038                 -0.813                    -1.045
1           1.151         1.248             1.198        1.255                  1.985                     1.162
2          -0.004        -0.168            -0.091       -0.142                 -0.288                    -0.096
3           1.115         1.224             1.181        1.210                  0.317                     1.184

Cluster Means (original scale):
            age  annual_spend  visits_per_month  basket_size  days_since_last_visit  num_categories_purchased
cluster                                                               

**Cluster Interpretation:**

- **Cluster 0 — Young Budget Shoppers (n=170):** Youngest customers (~25 yrs), lowest annual spend (~₹20K), least frequent visits, smallest basket. These are likely price-sensitive new customers or students.

- **Cluster 1 — High-Value Lapsed Customers (n=80):** Older (~57 yrs), high spenders (~₹85K), frequent buyers with large baskets — but haven't visited in a long time (75 days). These are dormant premium customers worth re-engaging.

- **Cluster 2 — Mid-Value Regulars (n=165):** Average on all metrics. Middle-aged (~40 yrs), moderate spend (~₹45K), visit regularly. The backbone of the customer base.

- **Cluster 3 — High-Value Active Loyalists (n=85):** Older (~57 yrs), high spenders (~₹84K), frequent visits, large baskets, and they visit regularly (36 days). This is the most valuable and engaged segment.

## 4. Dimensionality Reduction with PCA

In [5]:
# Apply PCA — reduce to 2 principal components
pca = PCA(n_components=2, random_state=42)
X_scaled_no_cluster = scaler.transform(df.drop('cluster', axis=1))
pcs = pca.fit_transform(X_scaled_no_cluster)

print('Explained Variance Ratio:')
for i, var in enumerate(pca.explained_variance_ratio_, 1):
    print(f'  PC{i}: {var:.4f} ({var*100:.2f}%)')
print(f'  Total: {sum(pca.explained_variance_ratio_)*100:.2f}%')

loadings = pd.DataFrame(
    pca.components_,
    columns=df.columns[:-1],
    index=['PC1', 'PC2']
)
print('\nFeature Loadings (components):')
print(loadings.round(4))

Explained Variance Ratio:
  PC1: 0.8356 (83.56%)
  PC2: 0.0557 (5.57%)
  Total: 89.13%

Feature Loadings (components):
       age  annual_spend  visits_per_month  basket_size  days_since_last_visit  num_categories_purchased
PC1  0.4116        0.4215            0.4113       0.4125                  0.3786                    0.4140
PC2 -0.2594       -0.0333           -0.0881      -0.0965                  0.9112                   -0.1405


**PC Interpretation:**

- **PC1 (83.6% variance):** All features load positively and roughly equally on PC1. It represents a **general customer value score** — customers high on PC1 are older, spend more, visit more often, buy larger baskets, and purchase across more categories.

- **PC2 (5.6% variance):** Dominated almost entirely by `days_since_last_visit` (loading: 0.91). It captures **recency of engagement** — high PC2 means a customer who hasn't visited in a long time, irrespective of their value level.

## 5. Cluster Visualisation

In [6]:
# Scatter plot: PC1 vs PC2 coloured by cluster
colors = ['#2196F3', '#F44336', '#4CAF50', '#FF9800']
cluster_names = ['Cluster 0: Young Budget', 'Cluster 1: Lapsed Premium', 'Cluster 2: Mid Regulars', 'Cluster 3: Active Loyalists']

plt.figure(figsize=(10, 7))
for c in range(4):
    mask = df['cluster'] == c
    plt.scatter(pcs[mask, 0], pcs[mask, 1], c=colors[c], label=cluster_names[c], alpha=0.7, edgecolors='white', s=60)

plt.title('Customer Segments — PCA Projection (PC1 vs PC2)', fontsize=13, fontweight='bold')
plt.xlabel('PC1 — Overall Customer Value (83.6% variance)')
plt.ylabel('PC2 — Recency of Visit (5.6% variance)')
plt.legend(loc='upper left', framealpha=0.9)
plt.tight_layout()
plt.savefig('cluster_pca.png', dpi=100, bbox_inches='tight')
plt.show()
print('PCA cluster plot saved.')

PCA cluster plot saved.
